# LAB | Imbalanced

**Load the data**

In this challenge, we will be working with Credit Card Fraud dataset.

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv

Metadata

- **distance_from_home:** the distance from home where the transaction happened.
- **distance_from_last_transaction:** the distance from last transaction happened.
- **ratio_to_median_purchase_price:** Ratio of purchased price transaction to median purchase price.
- **repeat_retailer:** Is the transaction happened from same retailer.
- **used_chip:** Is the transaction through chip (credit card).
- **used_pin_number:** Is the transaction happened by using PIN number.
- **online_order:** Is the transaction an online order.
- **fraud:** Is the transaction fraudulent. **0=legit** -  **1=fraud**


In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
fraud = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv")
fraud.head()

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,57.877857,0.311140,1.945940,1.0,1.0,0.0,0.0,0.0
1,10.829943,0.175592,1.294219,1.0,0.0,0.0,0.0,0.0
2,5.091079,0.805153,0.427715,1.0,0.0,0.0,1.0,0.0
3,2.247564,5.600044,0.362663,1.0,1.0,0.0,1.0,0.0
4,44.190936,0.566486,2.222767,1.0,1.0,0.0,1.0,0.0


**Steps:**

- **1.** What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?
- **2.** Train a LogisticRegression.
- **3.** Evaluate your model. Take in consideration class importance, and evaluate it by selection the correct metric.
- **4.** Run **Oversample** in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model? 
- **5.** Now, run **Undersample** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?
- **6.** Finally, run **SMOTE** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model? 

In [6]:
pip install imbalanced-learn

  Using cached imbalanced_learn-0.14.2-py3-none-any.whl.metadata (8.9 kB)
  Using cached sklearn_compat-0.1.6-py3-none-any.whl.metadata (22 kB)
Using cached imbalanced_learn-0.14.2-py3-none-any.whl (236 kB)
Using cached sklearn_compat-0.1.6-py3-none-any.whl (22 kB)

   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   ---------------------------------------- 2/2 [imbalanced-learn]

Note: you may need to restart the kernel to use updated packages.


In [7]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, average_precision_score, roc_auc_score
from sklearn.utils import resample
from imblearn.over_sampling import SMOTE

fraud = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv")
print(fraud.head())
print(fraud.shape, "nulls:", fraud.isna().sum().sum(), "dupes:", fraud.duplicated().sum())

   distance_from_home  distance_from_last_transaction  \
0           57.877857                        0.311140   
1           10.829943                        0.175592   
2            5.091079                        0.805153   
3            2.247564                        5.600044   
4           44.190936                        0.566486   

   ratio_to_median_purchase_price  repeat_retailer  used_chip  \
0                        1.945940              1.0        1.0   
1                        1.294219              1.0        0.0   
2                        0.427715              1.0        0.0   
3                        0.362663              1.0        1.0   
4                        2.222767              1.0        1.0   

   used_pin_number  online_order  fraud  
0              0.0           0.0    0.0  
1              0.0           0.0    0.0  
2              0.0           1.0    0.0  
3              0.0           1.0    0.0  
4              0.0           1.0    0.0  
(1000000, 8) n

In [8]:
# 1. Target distribution
print(fraud["fraud"].value_counts())
print(fraud["fraud"].value_counts(normalize=True))

X = fraud.drop(columns="fraud")
y = fraud["fraud"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

scaler = StandardScaler()
X_train_s = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test_s = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)


def evaluate(name, X_fit, y_fit):
    # 2-3. Train on (possibly resampled) train data, always evaluate on the untouched test set
    print(f"\n=== {name} ===")
    print("train distribution:", y_fit.value_counts().to_dict())
    model = LogisticRegression(max_iter=1000, random_state=42).fit(X_fit, y_fit)
    pred = model.predict(X_test_s)
    proba = model.predict_proba(X_test_s)[:, 1]
    print(classification_report(y_test, pred, digits=3))
    print(confusion_matrix(y_test, pred))
    row = classification_report(y_test, pred, output_dict=True)["1.0"]
    return {"model": name, "precision_1": row["precision"], "recall_1": row["recall"], "f1_1": row["f1-score"],
            "pr_auc": average_precision_score(y_test, proba), "roc_auc": roc_auc_score(y_test, proba),
            "fn": int(((y_test == 1) & (pred == 0)).sum()), "fp": int(((y_test == 0) & (pred == 1)).sum())}


results = [evaluate("Baseline", X_train_s, y_train)]

fraud
0.0    912597
1.0     87403
Name: count, dtype: int64
fraud
0.0    0.912597
1.0    0.087403
Name: proportion, dtype: float64

=== Baseline ===
train distribution: {0.0: 730078, 1.0: 69922}
              precision    recall  f1-score   support

         0.0      0.963     0.993     0.978    182519
         1.0      0.896     0.606     0.723     17481

    accuracy                          0.959    200000
   macro avg      0.930     0.799     0.850    200000
weighted avg      0.958     0.959     0.956    200000

[[181296   1223]
 [  6895  10586]]


In [9]:
# 4. Oversample minority (train only)
train = X_train_s.assign(fraud=y_train)
majority, minority = train[train["fraud"] == 0], train[train["fraud"] == 1]
over = pd.concat([majority, resample(minority, replace=True, n_samples=len(majority), random_state=42)])
results.append(evaluate("Oversample", over.drop(columns="fraud"), over["fraud"]))


=== Oversample ===
train distribution: {0.0: 730078, 1.0: 730078}
              precision    recall  f1-score   support

         0.0      0.995     0.934     0.963    182519
         1.0      0.577     0.948     0.718     17481

    accuracy                          0.935    200000
   macro avg      0.786     0.941     0.840    200000
weighted avg      0.958     0.935     0.942    200000

[[170391  12128]
 [   910  16571]]


In [10]:
# 5. Undersample majority (train only)
under = pd.concat([resample(majority, replace=False, n_samples=len(minority), random_state=42), minority])
results.append(evaluate("Undersample", under.drop(columns="fraud"), under["fraud"]))



=== Undersample ===
train distribution: {0.0: 69922, 1.0: 69922}
              precision    recall  f1-score   support

         0.0      0.995     0.934     0.963    182519
         1.0      0.577     0.947     0.717     17481

    accuracy                          0.935    200000
   macro avg      0.786     0.941     0.840    200000
weighted avg      0.958     0.935     0.942    200000

[[170394  12125]
 [   918  16563]]


In [11]:
# 6. SMOTE (train only)
X_sm, y_sm = SMOTE(random_state=42).fit_resample(X_train_s, y_train)
results.append(evaluate("SMOTE", X_sm, y_sm))

pd.set_option("display.width", 200)
print(pd.DataFrame(results).round(4).to_string(index=False))


=== SMOTE ===
train distribution: {0.0: 730078, 1.0: 730078}
              precision    recall  f1-score   support

         0.0      0.995     0.934     0.963    182519
         1.0      0.577     0.948     0.718     17481

    accuracy                          0.935    200000
   macro avg      0.786     0.941     0.840    200000
weighted avg      0.958     0.935     0.942    200000

[[170386  12133]
 [   907  16574]]
      model  precision_1  recall_1   f1_1  pr_auc  roc_auc   fn    fp
   Baseline       0.8964    0.6056 0.7228  0.8072   0.9670 6895  1223
 Oversample       0.5774    0.9479 0.7177  0.7574   0.9795  910 12128
Undersample       0.5773    0.9475 0.7175  0.7569   0.9796  918 12125
      SMOTE       0.5774    0.9481 0.7177  0.7574   0.9795  907 12133


**Does resampling improve the model?**

On recall, yes: recall rises from 0.61 to about 0.95, so missed fraud drops by about 87% (about 6,000 fewer cases).
The cost is false alarms: they rise about tenfold, from 1,223 to about 12,100, and precision falls from 0.90 to 0.58.

Overall quality is not clearly better: F1 is essentially flat, and PR-AUC actually drops from 0.807 to 0.757, while ROC-AUC rises slightly. Resampling mainly shifts the balance towards catching fraud rather than making a better model.

Which to choose: the three methods are practically tied. Undersampling is the practical pick because it trains on about 140,000 rows instead of 1.46 million. SMOTE adds nothing here.